# FRC Match Team Metrics
Displays the historical metrics of teams at an event.

## Setup
In your virtual environment, install pandas and matplotlib: 
  `pip install pandas matplotlib`
* If you are using VS Code, it should ask you to install the IPython extensions.
* If this next cell runs with no errors, you are all set.

In [1]:
import util
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

## Functions to get Team and Event names
Some parts of this notebook depend on a team you are interested in.  Other parts of this notebook need to look up the team name.  This section defines a function to look up the team name.  Same things for event name.

In [2]:
def get_team_name(team_id):
    url = f'https://www.thebluealliance.com/api/v3/team/{team_id}'
    return util.call_tba_api(url).json()['nickname']

def get_event_name(event_id):
    url = f'https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/simple'
    resp = util.call_tba_api(url).json()
    return str(resp['year']) + ' ' + resp['name']

## Default Team and Event

In [3]:
YEAR = '2025'
TEAM = 'frc6223'
EVENT_KEY = '2025wimu'
# EVENT_KEY = '2025wimi'

TEAM_NAME = get_team_name(TEAM)
EVENT_NAME = get_event_name(EVENT_KEY)

print(TEAM_NAME, 'at', EVENT_NAME)

Arsenal of Engineering at 2025 Phantom Lakes Regional


## Teams at the Event

In [4]:
url = f'https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/teams/simple'
resp = util.call_tba_api(url).json()
df = pd.DataFrame.from_dict(resp, orient='columns')
print('Team count: ', str(len(df)))
del url, resp
df.drop(columns=['country', 'name', 'team_number'], inplace=True)
df


Team count:  41


,city,key,nickname,state_prov
0,Menomonie,frc10264,STORM,Wisconsin
1,Oregon,frc10553,Orange Overdrive,Wisconsin
2,Hartford,frc1091,Oriole Assault,Wisconsin
3,Milwaukee,frc1220,Rockhoppers - Hilltopper Robotics,Wisconsin
4,Middleton,frc1306,BadgerBOTS,Wisconsin
5,Milwaukee,frc1714,MORE Robotics,Wisconsin
6,Milwaukee,frc1732,Hilltopper Robotics,Wisconsin
7,Oak Creek,frc1792,Round Table Robotics,Wisconsin
8,Waukesha,frc2062,CORE 2062,Wisconsin
9,Wales,frc2077,Laser Robotics,Wisconsin


## Loop through teams to get metrics

In [5]:
df2 = pd.DataFrame(columns=['key', 'events', 'qual_wins', 'qual_loss', 'qual_ties', 'poff_wins', 'poff_loss', 'poff_ties', 'ranking'])
print('Researching:', end="")
for team in df['key']:
    print('.', end="")
    
    matches = 0
    qual_wins = 0
    qual_loss = 0
    qual_ties = 0
    poff_wins = 0
    poff_loss = 0
    poff_ties = 0
    ranking = []
    
    
    url = f'https://www.thebluealliance.com/api/v3/team/{team}/events/{YEAR}/statuses'
    resp = util.call_tba_api(url).json()
    
    for match in resp:
        details = resp[match]
        
        if details != None and details['qual'] != None:
            matches += 1
            qual_wins += details['qual']['ranking']['record']['wins']
            qual_loss += details['qual']['ranking']['record']['losses']
            qual_ties += details['qual']['ranking']['record']['ties']
            if details['playoff'] != None:
                poff_wins += details['playoff']['record']['wins'] 
                poff_loss += details['playoff']['record']['losses'] 
                poff_ties += details['playoff']['record']['ties'] 
            ranking.append(details['qual']['ranking']['rank'])

    df2.loc[len(df2)] = [team, matches, qual_wins, qual_loss, qual_ties, poff_wins, poff_loss, poff_ties, 0 if ranking is None or len(ranking) == 0 else sum(num for num in ranking if num is not None) / len(ranking)]
    # print('  ', matches, qual_wins, qual_loss, qual_ties, poff_wins, poff_loss, poff_ties, ranking)
    
df = df.join(df2.set_index('key'), on='key')
del df2

Researching:.........................................

In [6]:
# Set all ranking = 0 to 100
df['ranking'] = df['ranking'].replace(0, 100)

# Calculate win rates
df['win_qual'] = df['qual_wins'] / (df['qual_wins'] + df['qual_loss'] + df['qual_ties'])
df['win_poff'] = df['poff_wins'] / (df['poff_wins'] + df['poff_loss'] + df['poff_ties'])

# Convert the decimal column to percentages
df['win_qual'] = df['win_qual'].apply(lambda x: f"{x * 100:.0f}%")
df['win_poff'] = df['win_poff'].apply(lambda x: f"{x * 100:.0f}%")

In [7]:
#df.sort_values(by=['ranking', 'win_qual'], ascending=[True,False], inplace=True, ignore_index=True)
df.sort_values(by=['win_qual'], ascending=[False], inplace=True, ignore_index=True)
df

,city,key,nickname,state_prov,events,qual_wins,qual_loss,qual_ties,poff_wins,poff_loss,poff_ties,ranking,win_qual,win_poff
0,Menomonie,frc10264,STORM,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
1,Mukwonago,frc930,Mukwonago BEARs,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
2,Milwaukee,frc1220,Rockhoppers - Hilltopper Robotics,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
3,Marinette,frc8803,MechaMarines,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
4,Racine,frc7900,Trial N' Terror,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
5,Oak Creek,frc1792,Round Table Robotics,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
6,Racine,frc6643,Walnuts and Bolts,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
7,Wales,frc2077,Laser Robotics,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
8,Brookfield,frc2202,BEAST Robotics,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%
9,Menomonee Falls,frc6223,Arsenal of Engineering,Wisconsin,0,0,0,0,0,0,0,100.0,nan%,nan%


In [8]:
url = f'https://www.thebluealliance.com/api/v3/event/{EVENT_KEY}/matches/simple'
resp = util.call_tba_api(url).json()
for match in resp:
    if match['match_number'] in [7, 15, 22, 30, 45, 54, 62, 68, 79, 89]:
        print('Match', match['match_number'])
        print('  Red:')
        for team in match['alliances']['red']['team_keys']:
            print('    ', team, df[df['key'] == team].index[0] + 1) 
        print('  Blue:')
        for team in match['alliances']['blue']['team_keys']:
            print('    ', team, df[df['key'] == team].index[0] + 1) 
    # print(match['key'], match['alliances']['red']['team_keys'], match['alliances']['blue']['team_keys'], match['winning_alliance'])